# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We load the raw data, apply our exclusions (impressions_90d > 0 and content_age_days >= 90), and construct our feature vector. We also create the target label `is_declining_label` from the raw `trend_direction` field.
- **Exclusions**: We exclude young pages (age < 90 days) since they are in a discovery phase, and pages with 0 impressions in 90 days since there is no trend to analyze.
- **Target**: `is_declining_label = (trend_direction == 'down')`.

In [1]:
# Load raw data and build clean dataset
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Locate root dir by searching for requirements.txt
root = Path(os.getcwd())
for _ in range(5):
    if (root / 'requirements.txt').exists():
        break
    root = root.parent

raw_path = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(raw_path)
print(f"Raw shape: {df.shape}")

# Exclusions
df_cleaned = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df_cleaned = df_cleaned.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print(f"Cleaned shape after exclusions: {df_cleaned.shape}")

# Create label
df_cleaned["is_declining_label"] = df_cleaned["trend_direction"].str.lower().eq("down").astype(int)
print(f"Base rate (declining proportion): {df_cleaned['is_declining_label'].mean():.4f}")

# Feature Engineering
df_cleaned["log_impressions_90d"] = np.log1p(df_cleaned["impressions_90d"])
df_cleaned["log_clicks_90d"] = np.log1p(df_cleaned["clicks_90d"])
df_cleaned["log_sessions_90d"] = np.log1p(df_cleaned["sessions_90d"])
df_cleaned["log_ai_sessions_90d"] = np.log1p(df_cleaned["ai_sessions_90d"])
df_cleaned["has_clicks"] = (df_cleaned["clicks_90d"] > 0).astype(int)
df_cleaned["has_ai_sessions"] = (df_cleaned["ai_sessions_90d"] > 0).astype(int)
df_cleaned["measurable_opportunity"] = (
    (df_cleaned["impressions_90d"] >= 100) & (df_cleaned["sessions_90d"] > 0)
).astype(int)


Raw shape: (30000, 44)
Cleaned shape after exclusions: (30000, 44)
Base rate (declining proportion): 0.5421


## 2. Feature notes (meaning, missing, categorical, available-when?)

For the features we model, we ensure they are available BEFORE the decision point (meaning they only contain historical data up to day 0) and we document how missing values are handled.
- **Search performance**: log_impressions_90d, log_clicks_90d, ctr, avg_position (filled with 0/defaults if missing).
- **Engagement**: engagement_rate, scroll_rate (filled with 0).
- **Categorical metadata**: content_type, main_intent, age_tier (filled with 'unknown' if missing).

In [2]:
# Inspect features and fill-in checks
import sys
if str(root) not in sys.path:
    sys.path.append(str(root))
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

print("Model Numeric Features:", MODEL_NUMERIC_FEATURES)
print("\nModel Categorical Features:", MODEL_CATEGORICAL_FEATURES)

# Check missing values
print("\nMissing values counts in modeled numeric features:")
print(df_cleaned[MODEL_NUMERIC_FEATURES].isna().sum())


Model Numeric Features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Model Categorical Features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Missing values counts in modeled numeric features:
search_volume             2468
competition               2468
cpc                       2468
word_count                7699
char_count                7699
log_impressions_90d          0
log_clicks_90d               0
log_sessions_90d             0
log_ai_sessions_90d          0
days_with_impressions        0
days_with_sessions           0
content_age_days             0
days_since_last_update       0
ctr                          0
avg

## 3. The leakage hunt

To prevent feature leakage (which artificially inflates model performance during offline testing but fails in production):
- We verify that the target label `is_declining_label`, and its components `trend_direction` and `trend_pct`, are strictly excluded from the feature matrices.
- We check feature correlations with the target to detect any suspicious near-perfect correlations (e.g., correlation absolute value close to 1.0).

In [3]:
# Leakage verification checks
leakage_candidates = ["trend_direction", "trend_pct", "is_declining_label"]
for feat in leakage_candidates:
    in_numeric = feat in MODEL_NUMERIC_FEATURES
    in_categorical = feat in MODEL_CATEGORICAL_FEATURES
    print(f"Is '{feat}' leaked in numeric features? {in_numeric}")
    print(f"Is '{feat}' leaked in categorical features? {in_categorical}")

# Compute correlation of numeric features with target
correlations = df_cleaned[MODEL_NUMERIC_FEATURES].corrwith(df_cleaned["is_declining_label"])
print("\nCorrelations of numeric features with the target label:")
print(correlations.sort_values(ascending=False))


Is 'trend_direction' leaked in numeric features? False
Is 'trend_direction' leaked in categorical features? False
Is 'trend_pct' leaked in numeric features? False
Is 'trend_pct' leaked in categorical features? False
Is 'is_declining_label' leaked in numeric features? False
Is 'is_declining_label' leaked in categorical features? False

Correlations of numeric features with the target label:
days_with_impressions     0.190055
log_impressions_90d       0.177473
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
log_sessions_90d          0.015270
log_clicks_90d            0.003469
ai_traffic_pct            0.002435
scroll_rate              -0.002958
log_ai_sessions_90d      -0.004304
competition              -0.009341
engagement_rate          -0.012743
cpc                      -0.016502
search_volume            -0.019103
days_with_sessions       -0.025055
avg_position             -0.029035
ctr                      -0.061911
content_age_

## 4. What I excluded and why

We document the columns present in the raw data that we chose to exclude from modeling and the rationale behind each exclusion:
- `trend_direction` & `trend_pct`: Direct leakage (target is derived from them).
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`: Direct components of trend calculation (high risk of leak).
- `content_id` & `client_id`: Hashed unique identifier and client grouping keys, not predictive features.

In [4]:
# Excluded columns analysis
excluded_cols = [
    col for col in df.columns 
    if col not in MODEL_NUMERIC_FEATURES and col not in MODEL_CATEGORICAL_FEATURES
    and col not in ["content_id", "client_id", "is_declining_label"]
]
print(f"Number of excluded columns: {len(excluded_cols)}")
print("\nList of some excluded columns:")
for col in sorted(excluded_cols):
    print(f"- {col}")


Number of excluded columns: 20

List of some excluded columns:
- age_tier_order
- ai_sessions_90d
- char_count_tier
- clicks_90d
- clicks_last_30d
- clicks_prev_30d
- engaged_sessions_90d
- impressions_90d
- impressions_last_30d
- impressions_prev_30d
- model_used
- pageviews_90d
- provider_used
- scroll_events_90d
- sessions_90d
- sessions_last_30d
- sessions_prev_30d
- trend_direction
- trend_pct
- users_90d


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.